# 嵌入超过模型最大上下文长度的文本

OpenAI 的嵌入模型无法嵌入超过最大长度的文本。最大长度因模型而异，并且以 _token_ 而非字符串长度来衡量。如果你不熟悉分词（tokenization），请参阅 [如何使用 tiktoken 计算 token 数](How_to_count_tokens_with_tiktoken.ipynb)。

本 notebook 展示如何处理超过模型最大上下文长度的文本。我们将以 `text-embedding-3-small` 的嵌入为例进行演示，但同样的思路也适用于其他模型和任务。要进一步了解嵌入，请查看 [OpenAI 嵌入指南](https://beta.openai.com/docs/guides/embeddings)。


## 1. 模型上下文长度

首先，我们选择模型，并定义一个从 API 获取嵌入的函数。

In [2]:
from openai import OpenAI
import os
import openai
from tenacity import retry, wait_random_exponential, stop_after_attempt, retry_if_not_exception_type

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", "<your OpenAI API key if not set as env var>"))

EMBEDDING_MODEL = 'text-embedding-3-small'
EMBEDDING_CTX_LENGTH = 8191
EMBEDDING_ENCODING = 'cl100k_base'

# let's make sure to not retry on an invalid request, because that is what we want to demonstrate
@retry(wait=wait_random_exponential(min=1, max=20), stop=stop_after_attempt(6), retry=retry_if_not_exception_type(openai.BadRequestError))
def get_embedding(text_or_tokens, model=EMBEDDING_MODEL):
    return client.embeddings.create(input=text_or_tokens, model=model).data[0].embedding

`text-embedding-3-small` 模型在 `cl100k_base` 编码下的上下文长度为 8191 个 token，可以看到超出这个限制会导致报错。

In [4]:
long_text = 'AGI ' * 5000
try:
    get_embedding(long_text)
except openai.BadRequestError as e:
    print(e)

Error code: 400 - {'error': {'message': "This model's maximum context length is 8192 tokens, however you requested 10001 tokens (10001 in your prompt; 0 for the completion). Please reduce your prompt; or completion length.", 'type': 'invalid_request_error', 'param': None, 'code': None}}


显然，我们希望避免这些错误，尤其是在以编程方式处理大量嵌入时。然而，我们仍可能遇到长度超过最大上下文长度的文本。下面我们介绍处理这类长文本的主要方法并给出示例代码：(1) 直接将文本截断到允许的最大长度；(2) 将文本分块，并分别嵌入每个分块。

## 1. 截断输入文本

最简单的方案是将输入文本截断到允许的最大长度。由于上下文长度以 token 计量，我们需要先对文本进行分词再截断。API 同时接受文本或 token 形式的输入，因此只要确保使用了正确的编码，就无需将 token 转换回字符串。下面是这样一个截断函数的示例。

In [3]:
import tiktoken

def truncate_text_tokens(text, encoding_name=EMBEDDING_ENCODING, max_tokens=EMBEDDING_CTX_LENGTH):
    """Truncate a string to have `max_tokens` according to the given encoding."""
    encoding = tiktoken.get_encoding(encoding_name)
    return encoding.encode(text)[:max_tokens]

之前的示例现在可以正常运行，不再报错。

In [4]:
truncated = truncate_text_tokens(long_text)
len(get_embedding(truncated))

1536

## 2. 对输入文本分块

截断虽然可行，但丢弃可能相关的文本是一个明显的缺点。另一种方法是将输入文本划分为多个分块，然后分别嵌入每个分块。之后，我们既可以单独使用各分块的嵌入，也可以用某种方式将它们合并，例如取平均（按每个分块的大小加权）。

我们将借用 [Python 官方文档中的 recipes](https://docs.python.org/3/library/itertools.html#itertools-recipes) 里的一个函数，用于将序列拆分为多个分块。

In [5]:
from itertools import islice

def batched(iterable, n):
    """Batch data into tuples of length n. The last batch may be shorter."""
    # batched('ABCDEFG', 3) --> ABC DEF G
    if n < 1:
        raise ValueError('n must be at least one')
    it = iter(iterable)
    while (batch := tuple(islice(it, n))):
        yield batch

现在我们定义一个函数，将字符串编码为 token，再将其拆分为多个分块。

In [6]:
def chunked_tokens(text, encoding_name, chunk_length):
    encoding = tiktoken.get_encoding(encoding_name)
    tokens = encoding.encode(text)
    chunks_iterator = batched(tokens, chunk_length)
    yield from chunks_iterator

最后，我们可以编写一个能安全处理嵌入请求的函数：即使输入文本超过最大上下文长度，它也会将输入 token 分块并分别嵌入每个分块。将 `average` 标志设为 `True` 会返回各分块嵌入的加权平均值，设为 `False` 则直接返回未经修改的分块嵌入列表。

In [7]:
import numpy as np


def len_safe_get_embedding(text, model=EMBEDDING_MODEL, max_tokens=EMBEDDING_CTX_LENGTH, encoding_name=EMBEDDING_ENCODING, average=True):
    chunk_embeddings = []
    chunk_lens = []
    for chunk in chunked_tokens(text, encoding_name=encoding_name, chunk_length=max_tokens):
        chunk_embeddings.append(get_embedding(chunk, model=model))
        chunk_lens.append(len(chunk))

    if average:
        chunk_embeddings = np.average(chunk_embeddings, axis=0, weights=chunk_lens)
        chunk_embeddings = chunk_embeddings / np.linalg.norm(chunk_embeddings)  # normalizes length to 1
        chunk_embeddings = chunk_embeddings.tolist()
    return chunk_embeddings

我们现在又一次能够处理长输入文本了。

In [8]:
average_embedding_vector = len_safe_get_embedding(long_text, average=True)
chunks_embedding_vectors = len_safe_get_embedding(long_text, average=False)

print(f"Setting average=True gives us a single {len(average_embedding_vector)}-dimensional embedding vector for our long text.")
print(f"Setting average=False gives us {len(chunks_embedding_vectors)} embedding vectors, one for each of the chunks.")


Setting average=True gives us a single 1536-dimensional embedding vector for our long text.
Setting average=False gives us 2 embedding vectors, one for each of the chunks.


在某些情况下，按段落边界或句子边界拆分分块可能更合理，有助于保留文本的含义。